# Lab: Web Scraping - Books to Scrape


## Paso 1 - Importar librerías

In [2]:
# Instalamos las librerías necesarias si no están en el entorno
#!pip install requests beautifulsoup4

# Librería para hacer peticiones HTTP
import requests

# Librería para parsear (leer y navegar) el HTML de la página
from bs4 import BeautifulSoup

# Librería para organizar los datos en un DataFrame
import pandas as pd

## Paso 2 - Explorar la estructura HTML del sitio


In [3]:
# Hacemos una petición de prueba a la primera página del catálogo
url_test = "https://books.toscrape.com/catalogue/page-1.html"
response_test = requests.get(url_test)

print(f"Código de respuesta: {response_test.status_code}")

# Parseamos el HTML
soup_test = BeautifulSoup(response_test.content, "html.parser")

# Encontramos todos los libros de la página
books_test = soup_test.find_all('article', class_='product_pod')
print(f"Libros encontrados en la primera página: {len(books_test)}")

Código de respuesta: 200
Libros encontrados en la primera página: 20


In [4]:
# Exploramos el primer libro del listado para verificar la extracción de datos básicos
first_book = books_test[0]

# Título
title = first_book.find('h3').find('a')['title']

# Precio: eliminamos el símbolo '£' y convertimos a float
price_text = first_book.find('p', class_='price_color').text
price = float(price_text.replace('£', '').replace('Â', '').strip())

# Calificación: convertimos de palabra a número con un diccionario
rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
rating_word = first_book.find('p', class_='star-rating')['class'][1]
rating = rating_map[rating_word]

# Link relativo a la página de detalle del libro
relative_link = first_book.find('h3').find('a')['href']

print(f"Título: {title}")
print(f"Precio: {price}")
print(f"Rating: {rating}")
print(f"Link relativo: {relative_link}")

Título: A Light in the Attic
Precio: 51.77
Rating: 3
Link relativo: a-light-in-the-attic_1000/index.html


In [5]:
# Exploramos la página de detalle del primer libro para ver los campos adicionales

# Construimos la URL completa del detalle
# Los links del listado son relativos (ej: '../../../libro/index.html')
# La URL base del catálogo es: https://books.toscrape.com/catalogue/
base_url = "https://books.toscrape.com/catalogue/"
detail_url = base_url + relative_link.replace('../', '')

print(f"URL de detalle: {detail_url}")

# Hacemos la petición a la página de detalle
detail_response = requests.get(detail_url)
detail_soup = BeautifulSoup(detail_response.content, "html.parser")

# Extraemos la tabla con UPC y Availability
table = detail_soup.find('table', class_='table table-striped')
rows = table.find_all('tr')

# UPC: primera fila de la tabla
upc = rows[0].find('td').text

# Availability: sexta fila de la tabla (índice 5)
availability = rows[5].find('td').text.strip()

# Género: tercer elemento del breadcrumb (Home > Categoría > Título)
breadcrumb = detail_soup.find('ul', class_='breadcrumb')
genre = breadcrumb.find_all('li')[2].text.strip()

# Descripción: del tag meta 'description'
desc_tag = detail_soup.find('meta', attrs={'name': 'description'})
description = desc_tag['content'].strip() if desc_tag else 'No description available'

print(f"UPC: {upc}")
print(f"Availability: {availability}")
print(f"Genre: {genre}")
print(f"Description: {description[:100]}...")

URL de detalle: https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
UPC: a897fe39b1053632
Availability: In stock (22 available)
Genre: Poetry
Description: It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and...


## Paso 3 - Funciones auxiliares


In [9]:
def get_book_details(detail_url):
   
    
    # Hacemos la petición a la página de detalle del libro
    response = requests.get(detail_url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Extraemos la tabla con información técnica del libro
    table = soup.find('table', class_='table table-striped')
    rows = table.find_all('tr')
    
    # UPC está en la primera fila de la tabla
    upc = rows[0].find('td').text
    
    # Availability está en la sexta fila (índice 5)
    availability = rows[5].find('td').text.strip()
    
    # Género: tercer elemento del breadcrumb (Home > Género > Título)
    breadcrumb = soup.find('ul', class_='breadcrumb')
    genre = breadcrumb.find_all('li')[2].text.strip()
    
    # Descripción desde el meta tag
    desc_tag = soup.find('meta', attrs={'name': 'description'})
    description = desc_tag['content'].strip() if desc_tag else 'No description available'
    
    return {
        'UPC': upc,
        'Genre': genre,
        'Availability': availability,
        'Description': description
    }

In [10]:
def scrape_page(page_url):
    
    
    # Diccionario para convertir la calificación de texto a número
    rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
    
    # Hacemos la petición HTTP a la página del catálogo
    response = requests.get(page_url)
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Encontramos todos los artículos (libros) de la página
    books = soup.find_all('article', class_='product_pod')
    
    # Lista donde guardaremos los datos básicos de cada libro
    books_data = []
    
    for book in books:
        
        # Título desde el atributo 'title' del link
        title = book.find('h3').find('a')['title']
        
        # Precio: limpiamos el símbolo y convertimos a float
        price_text = book.find('p', class_='price_color').text
        price = float(price_text.replace('£', '').replace('Â', '').strip())
        
        # Calificación: convertimos de texto a número
        rating_word = book.find('p', class_='star-rating')['class'][1]
        rating = rating_map[rating_word]
        
        # Link relativo a la página de detalle
        relative_link = book.find('h3').find('a')['href']
        
        # Construimos la URL completa del detalle
        detail_url = "https://books.toscrape.com/catalogue/" + relative_link.replace('../', '')
        
        books_data.append({
            'Title': title,
            'Price (£)': price,
            'Rating': rating,
            'detail_url': detail_url
        })
    
    return books_data

## Paso 4 - Función principal `scrape_books(min_rating, max_price)`


In [11]:
def scrape_books(min_rating, max_price):
    
    
    # Lista donde acumulamos los libros que pasan el filtro
    filtered_books = []
    
    # El sitio tiene 50 páginas en el catálogo
    total_pages = 50
    
    print(f"Iniciando scraping con filtros: rating >= {min_rating}, precio <= {max_price}£")
    print("-" * 60)
    
    # --- FASE 1: Recorrer el catálogo y filtrar libros candidatos ---
    for page_num in range(1, total_pages + 1):
        
        # Construimos la URL de cada página del catálogo
        page_url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
        
        # Extraemos los datos básicos de los libros de esta página
        page_books = scrape_page(page_url)
        
        # Filtramos los libros que cumplen los criterios de rating y precio
        for book in page_books:
            if book['Rating'] >= min_rating and book['Price (£)'] <= max_price:
                filtered_books.append(book)
        
        # Mostramos progreso cada 10 páginas
        if page_num % 10 == 0:
            print(f"Páginas procesadas: {page_num}/{total_pages} | Libros filtrados hasta ahora: {len(filtered_books)}")
    
    print(f"\nFase 1 completada. Libros que cumplen los criterios: {len(filtered_books)}")
    print("\nFase 2: Obteniendo detalles de cada libro filtrado...")
    print("-" * 60)
    
    # --- FASE 2: Visitar la página de detalle de cada libro filtrado ---
    all_books_data = []
    
    for i, book in enumerate(filtered_books):
        
        # Visitamos la página de detalle para obtener UPC, Genre, Availability, Description
        details = get_book_details(book['detail_url'])
        
        # Combinamos los datos básicos con los detalles en el orden de columnas esperado
        all_books_data.append({
            'UPC': details['UPC'],
            'Title': book['Title'],
            'Price (£)': book['Price (£)'],
            'Rating': book['Rating'],
            'Genre': details['Genre'],
            'Availability': details['Availability'],
            'Description': details['Description']
        })
        
        # Mostramos progreso cada 20 libros
        if (i + 1) % 20 == 0:
            print(f"Detalles obtenidos: {i + 1}/{len(filtered_books)}")
    
    # Creamos el DataFrame final a partir de la lista de diccionarios
    df = pd.DataFrame(all_books_data)
    
    print(f"\nScraping completado. Total de libros en el DataFrame: {len(df)}")
    
    return df

## Paso 5 - Ejecutar la función y explorar resultados

In [12]:
# Ejecutamos la función con los parámetros de filtrado
# Calificación mínima: 3 estrellas | Precio máximo: 20£
df_books = scrape_books(min_rating=3, max_price=20)

Iniciando scraping con filtros: rating >= 3, precio <= 20£
------------------------------------------------------------
Páginas procesadas: 10/50 | Libros filtrados hasta ahora: 22
Páginas procesadas: 20/50 | Libros filtrados hasta ahora: 50
Páginas procesadas: 30/50 | Libros filtrados hasta ahora: 74
Páginas procesadas: 40/50 | Libros filtrados hasta ahora: 96
Páginas procesadas: 50/50 | Libros filtrados hasta ahora: 116

Fase 1 completada. Libros que cumplen los criterios: 116

Fase 2: Obteniendo detalles de cada libro filtrado...
------------------------------------------------------------
Detalles obtenidos: 20/116
Detalles obtenidos: 40/116
Detalles obtenidos: 60/116
Detalles obtenidos: 80/116
Detalles obtenidos: 100/116

Scraping completado. Total de libros en el DataFrame: 116


In [13]:
# Revisamos las primeras filas del DataFrame resultante
df_books.head(10)

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,e72a5dfc7e9267b2,The Coming Woman: A Novel Based on the Life of...,17.93,3,Default,In stock (19 available),"""If you have a heart, if you have a soul, Kare..."
1,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
2,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
3,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
4,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
5,7ae099f3898e0209,Unicorn Tracks,18.78,3,Fantasy,In stock (16 available),After a savage attack drives her from her home...
6,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock (16 available),"Every summer, Rose goes with her mom and dad t..."
7,709822d0b5bcb7f4,Thirst,17.27,5,Fiction,In stock (16 available),"On a searing summer Friday, Eddie Chapman has ..."
8,0da9aa9d24677fc0,The Life-Changing Magic of Tidying Up: The Jap...,16.77,3,Nonfiction,In stock (16 available),Despite constant efforts to declutter your hom...
9,0fa6dceead7ce47a,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5,Sequential Art,In stock (16 available),THE LONG-AWAITED STORY OF FANGIRLS TAKING ON T...


In [14]:
# Verificamos la estructura del DataFrame
print(f"Dimensiones: {df_books.shape}")
print(f"\nColumnas: {list(df_books.columns)}")
print(f"\nTipos de datos:")
print(df_books.dtypes)
print(f"\nValores nulos:")
print(df_books.isnull().sum())

Dimensiones: (116, 7)

Columnas: ['UPC', 'Title', 'Price (£)', 'Rating', 'Genre', 'Availability', 'Description']

Tipos de datos:
UPC              object
Title            object
Price (£)       float64
Rating            int64
Genre            object
Availability     object
Description      object
dtype: object

Valores nulos:
UPC             0
Title           0
Price (£)       0
Rating          0
Genre           0
Availability    0
Description     0
dtype: int64


In [15]:
# Estadísticas básicas del DataFrame final
print("Distribución de calificaciones:")
print(df_books['Rating'].value_counts().sort_index())

print(f"\nRango de precios: {df_books['Price (£)'].min():.2f}£ - {df_books['Price (£)'].max():.2f}£")

print(f"\nTop 5 géneros más frecuentes:")
print(df_books['Genre'].value_counts().head())

Distribución de calificaciones:
Rating
3    41
4    33
5    42
Name: count, dtype: int64

Rango de precios: 10.00£ - 19.69£

Top 5 géneros más frecuentes:
Genre
Default           16
Sequential Art    13
Nonfiction        12
Fiction            9
Young Adult        7
Name: count, dtype: int64


In [16]:
# DataFrame final completo
df_books

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,e72a5dfc7e9267b2,The Coming Woman: A Novel Based on the Life of...,17.93,3,Default,In stock (19 available),"""If you have a heart, if you have a soul, Kare..."
1,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock (19 available),Aaron Ledbetter’s future had been planned out ...
2,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock (18 available),"In The Four Agreements, don Miguel Ruiz reveal..."
3,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock (18 available),A page-turning novel that is also an explorati...
4,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock (16 available),"More than thirty-five years ago, when the weat..."
...,...,...,...,...,...,...,...
111,29fc016c459aeb14,The Edge of Reason (Bridget Jones #2),19.18,4,Womens Fiction,In stock (1 available),Monday 27 January“7:15 a.m. Hurrah! The wilder...
112,3301af038a720587,The Complete Maus (Maus #1-2),10.64,3,Sequential Art,In stock (1 available),Combined for the first time here are Maus I: A...
113,18b4545a5ed15581,The Communist Manifesto,14.76,3,Add a comment,In stock (1 available),A rousing call to arms whose influence is stil...
114,3a9a7c5895e82646,Sister Sable (The Mad Queen #1),13.33,3,Fantasy,In stock (1 available),THE FIRST TENET OF THE WIND: Do not get caught...
